# Export CORINE Land Cover for Lithuania

Exports:
- **PNG rasters** (dashboard style)
- **GeoTIFF rasters** (`rasters/corine/geotiff/`) for web display – zoomable, no quality loss
- **CSV** per-class area counts, comparable to HILDA / LUCAS / HYDE / LUH2

Run all cells. Ensure the notebook kernel uses the same env where rasterio works (e.g. landcover2).

In [1]:
from pathlib import Path
import json
!pip install shapely
from shapely.geometry import shape
from shapely.ops import unary_union
import numpy as np
import pandas as pd
import rasterio
import rasterio.mask
import rasterio.warp
from rasterio.transform import from_bounds
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

BASE = Path(r"C:\Users\matas\Desktop\LEI\Data")
LT_GEOJSON = BASE / "lt_boundary_admin.json"

CORINE_TIFS = {
    1990: BASE / "Validation" / "Corine_LandCover_raster" / "States"
        / "u2000_clc1990_v2020_20u1_raster100m"
        / "u2000_clc1990_v2020_20u1_raster100m" / "DATA"
        / "U2000_CLC1990_V2020_20u1.tif",
    2000: BASE / "Validation" / "Corine_LandCover_raster" / "States"
        / "u2006_clc2000_v2020_20u1_raster100m"
        / "u2006_clc2000_v2020_20u1_raster100m" / "DATA"
        / "U2006_CLC2000_V2020_20u1.tif",
    2006: BASE / "Validation" / "Corine_LandCover_raster" / "States"
        / "u2012_clc2006_v2020_20u1_raster100m"
        / "u2012_clc2006_v2020_20u1_raster100m" / "DATA"
        / "U2012_CLC2006_V2020_20u1.tif",
    2012: BASE / "Validation" / "Corine_LandCover_raster" / "States"
        / "u2018_clc2012_v2020_20u1_raster100m"
        / "u2018_clc2012_v2020_20u1_raster100m" / "DATA"
        / "U2018_CLC2012_V2020_20u1.tif",
    2018: BASE / "Validation" / "Corine_LandCover_raster" / "States"
        / "u2018_clc2018_v2020_20u1_raster100m"
        / "u2018_clc2018_v2020_20u1_raster100m" / "DATA"
        / "U2018_CLC2018_V2020_20u1.tif",
}

OUT_RASTERS = BASE / "rasters" / "corine"
OUT_GEOTIFF = OUT_RASTERS / "geotiff"
OUT_CSV = BASE / "outputs" / "corine_lithuania_timeseries.csv"
OUT_RASTERS.mkdir(parents=True, exist_ok=True)
OUT_GEOTIFF.mkdir(parents=True, exist_ok=True)
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

# Load Lithuania polygon
with open(LT_GEOJSON, "r", encoding="utf-8") as f:
    lt_data = json.load(f)
geoms = [shape(feat["geometry"]) for feat in lt_data["features"]]
lt_geom = unary_union(geoms)
print("Loaded Lithuania boundary", lt_geom.geom_type)

Loaded Lithuania boundary MultiPolygon


In [2]:
def load_lt_mask_for_dataset(src):
    """Reproject LT polygon to match the raster CRS."""
    if src.crs is None:
        raise ValueError("CORINE raster has no CRS; cannot mask.")
    lt_proj = rasterio.warp.transform_geom(
        "EPSG:4326", src.crs, lt_geom.__geo_interface__
    )
    return lt_proj


def corine_to_five_classes(arr):
    """Map CORINE compact codes to 5 classes: 1=Water, 2=Wetlands, 3=Urban, 4=Agriculture, 5=Forest."""
    out = np.full(arr.shape, np.nan, dtype="float32")
    idx = arr.astype("int32")
    index_to_clc = np.array(
        [111, 112, 121, 122, 123, 124, 131, 132, 133, 141, 142,
         211, 212, 213, 221, 222, 223, 231, 241, 242, 243, 244,
         311, 312, 313, 321, 322, 323, 324, 331, 332, 333, 334, 335,
         411, 412, 421, 422, 423, 511, 512, 521, 522, 523, 999],
        dtype="int32",
    )
    clc = np.full_like(idx, 999)
    valid = (idx >= 1) & (idx <= len(index_to_clc))
    clc[valid] = index_to_clc[idx[valid] - 1]
    hundreds = (clc // 100) * 100
    out[hundreds == 500] = 1
    out[hundreds == 400] = 2
    out[hundreds == 100] = 3
    out[hundreds == 200] = 4
    out[hundreds == 300] = 5
    out[clc == 999] = np.nan
    return out


class_names = {1: "Water", 2: "Wetlands", 3: "Urban", 4: "Agriculture", 5: "Forest"}
colors = ["#4DA6FF", "#7B68EE", "#FF4D4D", "#FFD24D", "#228B22"]
cmap = ListedColormap(colors)
cmap.set_bad((0, 0, 0, 0))
norm = matplotlib.colors.Normalize(vmin=1, vmax=5)

In [ ]:
records = []

for year, tif_path in sorted(CORINE_TIFS.items()):
    if not tif_path.exists():
        print(f"Skipping CORINE {year}: {tif_path} does not exist.")
        continue

    print(f"Processing CORINE {year}...")

    with rasterio.open(tif_path) as src:
        lt_proj = load_lt_mask_for_dataset(src)
        data, transform = rasterio.mask.mask(
            src, [lt_proj], crop=True, nodata=src.nodata
        )
        arr_raw = data[0]

        dst_crs = "EPSG:4326"
        south, west = 53.45, 20.45
        north, east = 56.65, 26.75
        res = 0.005
        height = int(round((north - south) / res))
        width = int(round((east - west) / res))
        dst_transform = from_bounds(west, south, east, north, width, height)

        dst_arr = np.full(
            (height, width),
            src.nodata if src.nodata is not None else 0,
            dtype=arr_raw.dtype,
        )

        rasterio.warp.reproject(
            source=arr_raw,
            destination=dst_arr,
            src_transform=transform,
            src_crs=src.crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=rasterio.warp.Resampling.nearest,
        )
        arr_wgs84 = dst_arr

    arr_classes = corine_to_five_classes(arr_wgs84)

    flat = arr_classes[np.isfinite(arr_classes)].astype(int)
    if flat.size:
        uniq, cnts = np.unique(flat, return_counts=True)
        for cls_id, cnt in zip(uniq, cnts):
            cls_id = int(cls_id)
            cls_name = class_names.get(cls_id, f"class_{cls_id}")
            records.append((int(year), cls_id, cls_name, int(cnt)))

    rgba = cmap(norm(arr_classes))
    out_png = OUT_RASTERS / f"corine_{int(year)}.png"
    plt.imsave(out_png, rgba)
    print(f"  Saved {out_png}")

    arr_uint8 = np.where(np.isfinite(arr_classes), arr_classes.astype(np.uint8), 0)
    out_tif = OUT_GEOTIFF / f"corine_{int(year)}.tif"
    with rasterio.open(
        out_tif,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype=arr_uint8.dtype,
        crs=dst_crs,
        transform=dst_transform,
        nodata=0,
    ) as dst:
        dst.write(arr_uint8, 1)
    print(f"  Saved {out_tif}")

print("Done.")

Processing CORINE 1990...
  Saved C:\Users\matas\Desktop\LEI\Data\rasters\corine\corine_1990.png
  Saved C:\Users\matas\Desktop\LEI\Data\rasters\corine\geotiff\corine_1990.tif
Processing CORINE 2000...


C:\Users\matas\AppData\Local\Temp\ipykernel_12452\503980834.py:57: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = np.where(np.isfinite(arr_classes), arr_classes.astype(np.uint8), 0)


  Saved C:\Users\matas\Desktop\LEI\Data\rasters\corine\corine_2000.png
  Saved C:\Users\matas\Desktop\LEI\Data\rasters\corine\geotiff\corine_2000.tif
Processing CORINE 2006...


C:\Users\matas\AppData\Local\Temp\ipykernel_12452\503980834.py:57: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = np.where(np.isfinite(arr_classes), arr_classes.astype(np.uint8), 0)


  Saved C:\Users\matas\Desktop\LEI\Data\rasters\corine\corine_2006.png
  Saved C:\Users\matas\Desktop\LEI\Data\rasters\corine\geotiff\corine_2006.tif
Processing CORINE 2012...


C:\Users\matas\AppData\Local\Temp\ipykernel_12452\503980834.py:57: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = np.where(np.isfinite(arr_classes), arr_classes.astype(np.uint8), 0)


  Saved C:\Users\matas\Desktop\LEI\Data\rasters\corine\corine_2012.png
  Saved C:\Users\matas\Desktop\LEI\Data\rasters\corine\geotiff\corine_2012.tif
Processing CORINE 2018...


C:\Users\matas\AppData\Local\Temp\ipykernel_12452\503980834.py:57: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = np.where(np.isfinite(arr_classes), arr_classes.astype(np.uint8), 0)


  Saved C:\Users\matas\Desktop\LEI\Data\rasters\corine\corine_2018.png
  Saved C:\Users\matas\Desktop\LEI\Data\rasters\corine\geotiff\corine_2018.tif
Done.


C:\Users\matas\AppData\Local\Temp\ipykernel_12452\503980834.py:57: RuntimeWarning: invalid value encountered in cast
  arr_uint8 = np.where(np.isfinite(arr_classes), arr_classes.astype(np.uint8), 0)


In [4]:
df = pd.DataFrame(records, columns=["year", "class_id", "class_name", "count"])
df.to_csv(OUT_CSV, index=False)
print("Saved CSV:", OUT_CSV)
df.head(10)

Saved CSV: C:\Users\matas\Desktop\LEI\Data\outputs\corine_lithuania_timeseries.csv


,year,class_id,class_name,count
0,1990,1,Water,6891
1,1990,2,Wetlands,3192
2,1990,3,Urban,11972
3,1990,4,Agriculture,224459
4,1990,5,Forest,116801
5,2000,1,Water,6898
6,2000,2,Wetlands,3230
7,2000,3,Urban,11945
8,2000,4,Agriculture,223664
9,2000,5,Forest,117578
